<a href="https://colab.research.google.com/github/GuardinTheDev/Is-This-Text-Ai-/blob/Model-E%C4%9Fitimi/ModelE%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [2]:
from datasets import load_dataset, DatasetDict

print("Hugging Face Hub'dan 'Yunij/kaggle-comp-daigt' yükleniyor...")
# 'Yunij/kaggle-comp-daigt' veri seti yükleniyor.
raw_dataset = load_dataset('Yunij/kaggle-comp-daigt')

# Veri setinin bölümlerini kontrol edelim ve 'train' ile 'validation' olarak ayarlayalım.
# Eğer doğrudan 'train' ve 'test' bölümleri varsa bunları kullanırız.
# Eğer sadece 'train' varsa, onu böleriz.
if isinstance(raw_dataset, DatasetDict):
    if 'train' in raw_dataset and 'test' in raw_dataset:
        dataset = {
            'train': raw_dataset['train'],
            'validation': raw_dataset['test']
        }
    elif 'train' in raw_dataset:
        # Sadece 'train' bölümü varsa, bunu eğitim ve doğrulama olarak bölelim.
        print("Sadece 'train' bölümü bulundu, eğitim ve doğrulama olarak bölünüyor...")
        split_data = raw_dataset['train'].train_test_split(test_size=0.1, seed=42)
        dataset = {
            'train': split_data['train'],
            'validation': split_data['test']
        }
    else:
        raise ValueError("Veri setinde 'train' bölümü bulunamadı.")
else:
    # Eğer raw_dataset bir DatasetDict değilse ve sadece bir Dataset ise (örneğin sadece train split)
    print("Yüklenen veri seti tek bir split içeriyor, eğitim ve doğrulama olarak bölünüyor...")
    split_data = raw_dataset.train_test_split(test_size=0.1, seed=42)
    dataset = {
        'train': split_data['train'],
        'validation': split_data['test']
    }

print("Yunij/kaggle-comp-daigt Veri Seti Başarıyla Yüklendi ve Bölümleri Ayarlandı!")
print("Eğitim seti boyutu:", len(dataset['train']))
print("Doğrulama seti boyutu:", len(dataset['validation']))
print("\nÖrnek bir veri (eğitim setinden):", dataset['train'][0])
print("\nÖrnek bir veri (doğrulama setinden):", dataset['validation'][0])

Hugging Face Hub'dan 'Yunij/kaggle-comp-daigt' yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/134M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44868 [00:00<?, ? examples/s]

Sadece 'train' bölümü bulundu, eğitim ve doğrulama olarak bölünüyor...
Yunij/kaggle-comp-daigt Veri Seti Başarıyla Yüklendi ve Bölümleri Ayarlandı!
Eğitim seti boyutu: 40381
Doğrulama seti boyutu: 4487

Örnek bir veri (eğitim setinden): {'text': 'The "Face on Mars" is Really Just a Natural Landform\n\nMany people see the iconic landform photographed by the Viking spacecraft on Mars in 1976 and believe that it must be an artificial structure created by intelligent lifeforms. However, upon closer examination of the evidence, it becomes clear that the "Face on Mars" is simply a naturally formed mesa and hill. \n\nOne of the main pieces of evidence that the landform is natural comes from higher resolution images taken by later Mars missions. These images reveal that the supposed "eyes" and "mouth" of the face are merely valleys and shadows caused by erosion over geological timescales. The fine detail needed to clearly define facial features is not present. Instead, we see uneven forms that

In [3]:
from transformers import AutoTokenizer

# Model altyapısını hazır veri setine bağlama adımı
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fonksiyonu(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

# Hafızadaki yerel veri setimizin 'train' ve 'validation' bölümlerini tokenlaştırıyoruz
tokenized_datasets = {
    'train': dataset['train'].map(tokenize_fonksiyonu, batched=True),
    'validation': dataset['validation'].map(tokenize_fonksiyonu, batched=True)
}
print("Tokenlaştırma tamamlandı, model eğitimine hazır!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/40381 [00:00<?, ? examples/s]

Map:   0%|          | 0/4487 [00:00<?, ? examples/s]

Tokenlaştırma tamamlandı, model eğitimine hazır!


In [4]:
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Cihaz Kontrolü (Colab'da T4 GPU seçiliyse 'cuda' aktif olur, yoksa 'cpu' çalışır)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Eğitim için kullanılan cihaz: {device}")

# 2. Modeli Sınıflandırma İçin Yüklüyoruz
# İnsan (0) ve Yapay Zeka (1) olmak üzere 2 sınıfımız olduğu için num_labels=2 yapıyoruz
model_name = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# 3. Model Başarısını Ölçmek İçin Metrik Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

# 4. Eğitim Hiperparametreleri (Ayarları)
training_args = TrainingArguments(
    output_dir="./ai_detector_local_results", # Çıktıların kaydedileceği klasör
    learning_rate=2e-5,                       # Transformer modelleri için ideal öğrenme oranı
    per_device_train_batch_size=16,            # Örnek sayımız çok az olduğu için küçük tuttuk
    per_device_eval_batch_size=16,
    num_train_epochs=3,                       # Modelin veriyi kaç tur döneceği (Epoch)
    weight_decay=0.01,
    eval_strategy="epoch",                    # Her epoch sonunda doğruluğu test et
    save_strategy="epoch",
    load_best_model_at_end=True,              # En başarılı modeli hafızada tut
    logging_steps=1,                          # Logları hemen görmek için 1 yaptık
    report_to="none"                          # Harici raporlama araçlarını kapat
)

# 5. Trainer (Eğitici) Kurulumu
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
)

# 6. Eğitimi Başlatıyoruz
print("\n--- Model Eğitimi Başlıyor ---")
trainer.train()

# 7. Modeli ve Tokenizer'ı Yerel Klasöre Kaydetme
drive_kayit_yolu = "/content/drive/MyDrive/en_iyi_detektor_modeli"

trainer.save_model(drive_kayit_yolu)
tokenizer.save_pretrained(drive_kayit_yolu)
print(f"\nEğitim tamamlandı ve model Google Drive'ınıza ({drive_kayit_yolu}) başarıyla kaydedildi!")
#trainer.save_model("./en_iyi_detektor_modeli")
#tokenizer.save_pretrained("./en_iyi_detektor_modeli")
#print("\nEğitim tamamlandı ve model './en_iyi_detektor_modeli' klasörüne kaydedildi!")

Eğitim için kullanılan cihaz: cuda


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Model Eğitimi Başlıyor ---


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Eğitim tamamlandı ve model Google Drive'ınıza (/content/drive/MyDrive/en_iyi_detektor_modeli) başarıyla kaydedildi!


In [9]:


import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Modelin Google Drive'daki klasör yolu
model_path = "/content/drive/MyDrive/en_iyi_detektor_modeli"

# Eğer Drive bağlı değilse otomatik bağlamaya çalışsın
if not os.path.exists("/content/drive"):
    print("Drive bağlı değil, bağlanılıyor...")
    from google.colab import drive
    drive.mount('/content/drive')

# Klasör kontrolü
if not os.path.exists(model_path):
    print(f"HATA: Google Drive'da '{model_path}' klasörü bulunamadı!")
    print("Lütfen önce modeli eğittiğinizden ve Drive'a kaydettiğinizden emin olun.")
else:
    print("Model Google Drive'dan yükleniyor, lütfen bekleyin...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model ve Tokenizer Yükleme
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval() # Modeli test moduna alıyoruz
    print("Model başarıyla yüklendi! Analize hazır.\n")

    def metni_analiz_et(metin):
        inputs = tokenizer(metin, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        olasiliklar = torch.softmax(logits, dim=-1)[0]
        tahmin_id = torch.argmax(logits, dim=-1).item()

        siniflar = {0: "İnsan Tarafından Yazılmış", 1: "Yapay Zeka Tarafından Yazılmış"}

        insan_skoru = olasiliklar[0].item() * 100
        yz_skoru = olasiliklar[1].item() * 100

        return siniflar[tahmin_id], insan_skoru, yz_skoru

    # --- Canlı Test Ekranı ---
    print("=== YAPAY ZEKA METİN DETEKTÖRÜ ===")
    print("Çıkış yapmak için küçük 'q' harfi yazıp Enter'a basın.")

    while True:
        kullanici_metni = input("\nAnaliz edilecek metni girin:\n> ")

        if kullanici_metni.strip().lower() == 'q':
            print("Program kapatıldı.")
            break

        if not kullanici_metni.strip():
            print("Lütfen boş bırakmayın.")
            continue

        karar, insan_yuzde, yz_yuzde = metni_analiz_et(kullanici_metni)

        print("\n" + "="*40)
        print(f"📊 SONUÇ: {karar}")
        print(f"👨‍💻 İnsan Yazısı İhtimali: %{insan_yuzde:.2f}")
        print(f"🤖 Yapay Zeka İhtimali: %{yz_yuzde:.2f}")
        print("="*40)


Model Google Drive'dan yükleniyor, lütfen bekleyin...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model başarıyla yüklendi! Analize hazır.

=== YAPAY ZEKA METİN DETEKTÖRÜ ===
Çıkış yapmak için küçük 'q' harfi yazıp Enter'a basın.

Analiz edilecek metni girin:
> Arkadaşlar selam! Bugün başıma gelen inanılmaz komik bir şeyi anlatmasam çatlarım. Sabah aceleyle evden çıktım, otobüsü kaçırmamak için resmen depar atıyorum. Durağa tam yaklaştım, otobüsün kapısı kapandı kapanacak derken şoför beni gördü ve bekledi. İçeri kendimi nasıl attığımı bilmiyorum, nefes nefese kalmışım. Tam akbili basacağım, bir baktım cüzdan yok! Meğer aceleden diğer montun cebinde unutmuşum. Rezilliğin boyutunu düşünebiliyor musunuz? Arkamda koca bir kuyruk, şoför yüzüme bakıyor... Tam 'Kaptan ben ineyim o zaman' diyecekken arkalardan bir öğrenci arkadaş 'Ben basarım abi, lafı bile olmaz' dedi. O an o çocuğa nasıl sarılmak istedim anlatamam. Gerçekten hala böyle güzel, ince düşünceli insanların olduğunu bilmek insanın gününü bir anda güzelleştiriyor. İçimde sabahtan beri o olayın verdiği harika bir enerji var, si